**Sondondo Valley —**

 **Phase 1: Persona Deduplication**
*    Probabilistic Record Linkage with Splink + DuckDB
*   Input:    personas.csv  (~47,072 rows, 17 columns)
*   Output:   personas_clustered_iter3.csv












**Pipeline overview**
1. Install dependencies & load data
2. Column & data quality checks
3. Data cleaning  
4. Blocking rule size analysis
5. Define comparisons
6. Build Linker
7. Train model (u via random sampling, m via EM × 2)
8. Predict
9. Cluster (threshold = 0.99 0r 0.95)
10. Quality checks on clusters
11. Export



**1: Install & load**

In [ ]:
import sys
# !{sys.executable} -m pip install splink
!pip install -q splink --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 75.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()   # select personas.csv when prompted

df = pd.read_csv("personas.csv", low_memory=False)

print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
df.head(10)

Saving personas.csv to personas (1).csv
Shape: (47072, 17)

Column dtypes:
event_idno              object
original_identifier     object
persona_type            object
name                    object
birth_place             object
birth_date              object
legitimacy_status       object
birth_date_precision    object
lastname                object
persona_idno            object
social_condition        object
marital_status          object
resident_in             object
death_place             object
death_date              object
death_date_precision    object
gender                  object
dtype: object


,event_idno,original_identifier,persona_type,name,birth_place,birth_date,legitimacy_status,birth_date_precision,lastname,persona_idno,social_condition,marital_status,resident_in,death_place,death_date,death_date_precision,gender
0,bautizo-1,APAucará-LB-L001_B001,baptized,domingo,NaN,1790-08-04,legitimo,exact,ayquipa,persona-1,NaN,NaN,NaN,NaN,NaN,NaN,male
1,bautizo-1,APAucará-LB-L001_B001,father,lucas,NaN,NaN,NaN,NaN,ayquipa,persona-2,NaN,NaN,NaN,NaN,NaN,NaN,male
2,bautizo-1,APAucará-LB-L001_B001,mother,sevastiana,NaN,NaN,NaN,NaN,quispe,persona-3,NaN,NaN,NaN,NaN,NaN,NaN,female
3,bautizo-1,APAucará-LB-L001_B001,godfather,vicente,NaN,NaN,NaN,NaN,guamani,persona-4,NaN,NaN,NaN,NaN,NaN,NaN,male
4,bautizo-2,APAucará-LB-L001_B002,baptized,dominga,NaN,1790-08-04,legitimo,exact,lulia,persona-5,NaN,NaN,NaN,NaN,NaN,NaN,female
5,bautizo-2,APAucará-LB-L001_B002,father,juan,NaN,NaN,NaN,NaN,lulia,persona-6,NaN,NaN,NaN,NaN,NaN,NaN,male
6,bautizo-2,APAucará-LB-L001_B002,mother,jospha,NaN,NaN,NaN,NaN,gomes,persona-7,NaN,NaN,NaN,NaN,NaN,NaN,unknown
7,bautizo-2,APAucará-LB-L001_B002,godfather,ignacio,NaN,NaN,NaN,NaN,varientos,persona-8,NaN,NaN,NaN,NaN,NaN,NaN,male
8,bautizo-3,APAucará-LB-L001_B003,baptized,bartola,NaN,1790-08-04,legitimo,exact,quispe,persona-9,NaN,NaN,NaN,NaN,NaN,NaN,female
9,bautizo-3,APAucará-LB-L001_B003,father,jacinto,NaN,NaN,NaN,NaN,quispe,persona-10,NaN,NaN,NaN,NaN,NaN,NaN,male


**2: Data quality checks**

In [ ]:
# 2a: Unique ID check
print("persona_idno is unique:", df['persona_idno'].is_unique)
print()

# 2b: Gender value distribution
print("Raw gender values:")
print(df['gender'].value_counts(dropna=False))
print()

# 2c: Name / lastname coverage
print("Missing 'name'    :", df['name'].isna().sum())
print("Missing 'lastname':", df['lastname'].isna().sum())
print()

# 2d: Case check (names should already be lowercase)
print("'name' is all-lowercase    :", df['name'].str.islower().mean())
print("'lastname' is all-lowercase:", df['lastname'].str.islower().mean())
print()

# 2e: Date range sanity check
birth_temp = pd.to_datetime(df['birth_date'], errors='coerce')
print("birth_date range:", birth_temp.min(), "→", birth_temp.max())


persona_idno is unique: True

Raw gender values:
gender
male             20150
female           16393
unknown           9618
andy               521
mostly_female      288
mostly_male        102
Name: count, dtype: int64

Missing 'name'    : 73
Missing 'lastname': 310

'name' is all-lowercase    : 1.0
'lastname' is all-lowercase: 1.0

birth_date range: 1742-06-25 00:00:00 → 1921-03-01 00:00:00


**3: Data cleaning (upfront, once)**

Fixes applied:
1. Convert date strings → proper datetime objects.
2. Collapse gender into three values: `male` /`female` / `None` (missing).
   `unknown` and `andy` (ambiguous/unisex) are set to None so Splink skips
   that field rather than treating two "unknown" people as a gender match.
3. Replace the `"en blanco"` placeholder with None.
   Transcribers wrote "en blanco" (Spanish for "blank") when the name was
   illegible, instead of leaving the cell empty. This caused a huge false
   cluster in Iteration 1 — 13 unrelated rows merged because "en blanco"
   matched "en blanco" exactly. Mapping to None fixes that.

In [ ]:
# 3a: Convert dates
df['birth_date'] = pd.to_datetime(df['birth_date'], errors='coerce')
df['death_date'] = pd.to_datetime(df['death_date'], errors='coerce')

# 3b: Collapse gender
gender_map = {
    'male':         'male',
    'mostly_male':  'male',     # fold into male — still useful signal
    'female':       'female',
    'mostly_female':'female',   # fold into female
    'unknown':      None,       # no info — skip rather than false-match
    'andy':         None,       # ambiguous name — no reliable gender signal
}
df['gender_clean'] = df['gender'].map(gender_map)

# 3c: Remove "en blanco" placeholder
df['name']     = df['name'].replace('en blanco', None)
df['lastname'] = df['lastname'].replace('en blanco', None)

# 3d: Sanity checks
print("gender_clean distribution:")
print(df['gender_clean'].value_counts(dropna=False))
print()
print("'en blanco' remaining — name:", (df['name'] == 'en blanco').sum(),
      " | lastname:", (df['lastname'] == 'en blanco').sum())

gender_clean distribution:
gender_clean
male      20252
female    16681
None      10139
Name: count, dtype: int64

'en blanco' remaining — name: 0  | lastname: 0


**4: Blocking rule size analysis**

1.   Blocking rules decide which pairs of rows Splink even considers comparing.
2.  Too broad → millions of pairs, very slow. Too tight → misses real matches.
3. We test three candidate rules and measure how many comparisons each generates.


In [ ]:
import sys
!{sys.executable} -m pip install splink

from splink import DuckDBAPI, block_on
from splink.blocking_analysis import count_comparisons_from_blocking_rule

db_api = DuckDBAPI()

rules_to_test = [
    block_on("lastname"),
    block_on("lastname", "name"),
    block_on("lastname", "substr(name,1,1)"),
    block_on("lastname", "substr(name,1,1)", "gender_clean"),
]

for r in rules_to_test:
    result = count_comparisons_from_blocking_rule(
        table_or_tables=df,
        blocking_rule=r,
        link_type="dedupe_only",
        db_api=db_api,
        unique_id_column_name="persona_idno",
    )
    print(r, "→", result)

<splink.internals.blocking_rule_library.ExactMatchRule object at 0x7dfe8b23d5b0> → {'number_of_comparisons_generated_pre_filter_conditions': 26191064, 'number_of_comparisons_to_be_scored_post_filter_conditions': 13072161, 'filter_conditions_identified': '', 'equi_join_conditions_identified': 'l."lastname" = r."lastname"', 'link_type_join_condition': 'where l."persona_idno" < r."persona_idno"'}
<splink.internals.blocking_rule_library.And object at 0x7dfe8b433e30> → {'number_of_comparisons_generated_pre_filter_conditions': 348402, 'number_of_comparisons_to_be_scored_post_filter_conditions': 150867, 'filter_conditions_identified': '', 'equi_join_conditions_identified': 'l."lastname" = r."lastname" AND l."name" = r."name"', 'link_type_join_condition': 'where l."persona_idno" < r."persona_idno"'}
<splink.internals.blocking_rule_library.And object at 0x7dfe8b245e50> → {'number_of_comparisons_generated_pre_filter_conditions': 2341728, 'number_of_comparisons_to_be_scored_post_filter_conditions

**5: Define comparisons**

In [ ]:
# 5: Define comparisons (with term frequency adjustment)

import splink.comparison_library as cl

# Name & lastname: fuzzy (Jaro-Winkler) + term frequency penalty for common names
name_comparison     = cl.NameComparison("name").configure(term_frequency_adjustments=True)
lastname_comparison = cl.NameComparison("lastname").configure(term_frequency_adjustments=True)

# Gender: exact match only — 2 values, no fuzzy needed; nulls skipped
gender_comparison = cl.ExactMatch("gender_clean")

# Birth place: exact match — NER-derived, so spelling should be standardised;
# place data is considered lower-reliability (supervisor flag), so used cautiously
place_comparison = cl.ExactMatch("birth_place")

# Birth date: graduated comparison — exact > within 1 day > 1 month > 1 year
# (death_date excluded: only filled for "deceased" role rows, too sparse to train on)
date_comparison = cl.DateOfBirthComparison(
    "birth_date",
    input_is_string=False,
    datetime_metrics=["day", "month", "year"],
    datetime_thresholds=[1, 1, 1],
)

# Quick check — print what Splink built for name comparison
print(name_comparison.get_comparison("duckdb").human_readable_description)


Comparison 'NameComparison' of "name".
Similarity is assessed using the following ComparisonLevels:
    - 'name is NULL' with SQL rule: "name_l" IS NULL OR "name_r" IS NULL
    - 'Exact match on name' with SQL rule: "name_l" = "name_r"
    - 'Jaro-Winkler distance of name >= 0.92' with SQL rule: jaro_winkler_similarity("name_l", "name_r") >= 0.92
    - 'Jaro-Winkler distance of name >= 0.88' with SQL rule: jaro_winkler_similarity("name_l", "name_r") >= 0.88
    - 'Jaro-Winkler distance of name >= 0.7' with SQL rule: jaro_winkler_similarity("name_l", "name_r") >= 0.7
    - 'All other comparisons' with SQL rule: ELSE



**6: Build Linker**

**- The blocking rules, comparisons, and data into a single Splink Linker object.)**

In [ ]:
from splink import Linker, SettingsCreator

blocking_rules = [
    block_on("lastname", "substr(name,1,1)", "gender_clean"),  # tight: has gender
    block_on("lastname", "substr(name,1,1)"),                  # fallback: missing gender
]

settings = SettingsCreator(
    link_type="dedupe_only",
    unique_id_column_name="persona_idno",
    blocking_rules_to_generate_predictions=blocking_rules,
    comparisons=[
        name_comparison,
        lastname_comparison,
        gender_comparison,
        date_comparison,
        place_comparison,
    ],
    retain_intermediate_calculation_columns=True,
)

linker = Linker(df, settings, db_api=db_api)
print("Linker built successfully")

Linker built successfully


**7: Train model**

Splink needs to estimate two sets of probabilities:

1. **u probabilities** — how often two *random, unrelated* records look similar on each field (estimated via random sampling).
2. **m probabilities** — how often two records that *are* the same real person look similar on each field (estimated via Expectation Maximisation).
3. Two EM sessions are used to give Splink different "views" of the data:
a. One blocking on name (trains name/lastname well),
b. one blocking on birth_date (trains date comparison well).
The seed makes runs reproducible.

In [ ]:
# Train model - u probabilities (random sampling)

linker.training.estimate_u_using_random_sampling(max_pairs=1e7, seed=42)

INFO:splink.internals.estimate_u:----- Estimating u probabilities using random sampling -----


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - name (no m values are trained).
    - lastname (no m values are trained).
    - gender_clean (no m values are trained).
    - birth_date (no m values are trained).
    - birth_place (no m values are trained).


In [ ]:
# Train m probabilities - EM session 1 (name/lastname)

training_rule_1 = block_on("lastname", "substr(name,1,1)")
linker.training.estimate_parameters_using_expectation_maximisation(training_rule_1)

INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
(l."lastname" = r."lastname") AND (SUBSTR(l.name, 1, 1) = SUBSTR(r.name, 1, 1))

Parameter estimates will be made for the following comparison(s):
    - gender_clean
    - birth_date
    - birth_place

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - name
    - lastname
INFO:splink.internals.expectation_maximisation:
INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was -0.921 in the m_probability of birth_date, level `Exact match on date of birth`
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was 0.378 in the m_probability of birth_date, level `All other comparisons`
INFO:splink.internals.expectation_maximisation:Iteration 3: Largest change in params was 0.0557 

<EMTrainingSession, blocking on (l."lastname" = r."lastname") AND (SUBSTR(l.name, 1, 1) = SUBSTR(r.name, 1, 1)), deactivating comparisons name, lastname>

In [ ]:
# Train m probabilities — EM session 2 (dates)

training_rule_2 = block_on("birth_date")
linker.training.estimate_parameters_using_expectation_maximisation(training_rule_2)

INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
l."birth_date" = r."birth_date"

Parameter estimates will be made for the following comparison(s):
    - name
    - lastname
    - gender_clean
    - birth_place

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - birth_date
INFO:splink.internals.expectation_maximisation:
INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was 0.325 in the m_probability of gender_clean, level `Exact match on gender_clean`
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was -0.424 in the m_probability of lastname, level `Exact match on lastname`
INFO:splink.internals.expectation_maximisation:Iteration 3: Largest change in params was -0.0895 in the m_probability of lastname, level `Exact

<EMTrainingSession, blocking on l."birth_date" = r."birth_date", deactivating comparisons birth_date>

**8: Predict**

**- Score all candidate pairs generated by the blocking rules. Each pair receives a `match_probability` between 0 and 1.**

In [ ]:
predictions = linker.inference.predict()
df_predictions = predictions.as_pandas_dataframe()

print("Total candidate pairs scored:", df_predictions.shape[0])
df_predictions.sort_values("match_probability", ascending=False).head(10)

INFO:splink.internals.linker_components.inference:Blocking time: 0.58 seconds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

INFO:splink.internals.linker_components.inference:Predict time: 3.82 seconds


Total candidate pairs scored: 1147530


,match_weight,match_probability,persona_idno_l,persona_idno_r,name_l,name_r,gamma_name,tf_name_l,tf_name_r,bf_name,...,bf_gender_clean,birth_date_l,birth_date_r,gamma_birth_date,bf_birth_date,birth_place_l,birth_place_r,gamma_birth_place,bf_birth_place,match_key
784345,11.606175,0.999679,persona-30011,persona-30165,maría narcisa,maría narcisa,4,0.000043,0.000043,59.787039,...,1.30432,NaT,NaT,-1,1.000000,None,None,-1,1.00000,0
42297,11.021213,0.999519,persona-7401,persona-7415,maria ancelma,maria ancelma,4,0.000043,0.000043,59.787039,...,1.30432,NaT,NaT,-1,1.000000,None,None,-1,1.00000,0
783815,10.284247,0.999199,persona-29002,persona-29012,jose ma maria,jose ma maria,4,0.000043,0.000043,59.787039,...,1.30432,NaT,NaT,-1,1.000000,None,None,-1,1.00000,0
974003,10.222878,0.999164,persona-22799,persona-22823,innoto,innoto,4,0.000043,0.000043,59.787039,...,1.00000,NaT,NaT,-1,1.000000,None,None,-1,1.00000,1
846,10.021213,0.999039,persona-2005,persona-2420,benito joseph,benito joseph,4,0.000043,0.000043,59.787039,...,1.30432,NaT,NaT,-1,1.000000,None,None,-1,1.00000,0
293899,9.676603,0.998780,persona-12108,persona-12112,brijida,brijida,4,0.001383,0.001383,59.787039,...,1.00000,1902-10-07,1902-10-07,5,39.216401,pampamarca,pampamarca,1,3.26316,1
896908,9.637915,0.998746,persona-15311,persona-45563,fructuosa,fructuosa,4,0.000128,0.000128,59.787039,...,1.00000,NaT,1826-12-30,-1,1.000000,None,lucanas,-1,1.00000,1
1117380,9.460150,0.998582,persona-42350,persona-42491,lauriana,lauriana,4,0.000192,0.000192,59.787039,...,1.00000,1867-11-09,1867-11-09,5,39.216401,pampamarca,pampamarca,1,3.26316,1
897398,9.222878,0.998329,persona-16235,persona-16515,xisto h,xisto h,4,0.000043,0.000043,59.787039,...,1.00000,NaT,NaT,-1,1.000000,None,None,-1,1.00000,1
807293,9.213858,0.998319,persona-31023,persona-41131,florencio,florencio,4,0.000149,0.000149,59.787039,...,1.30432,NaT,1876-04-03,-1,1.000000,None,huanta,-1,1.00000,0


In [ ]:
print("\nTF columns present:", [c for c in df_predictions.columns if 'tf_' in c])


TF columns present: ['tf_name_l', 'tf_name_r', 'bf_tf_adj_name', 'tf_lastname_l', 'tf_lastname_r', 'bf_tf_adj_lastname']


**9: Cluster**

**Notes:**
1. Clustering groups pairs into connected components — all rows that transitively link to each other above the threshold are assigned the same `cluster_id`.
2. **Threshold = 0.99** (raised from 0.95 in Iteration 1 and 2).
Reason: at 0.95, too many name-only clusters (no date or place support) were produced — these are plausible but hard to verify.
At 0.99, only very high-confidence merges happen, reducing false positives at the cost of some recall. Acceptable trade-off for historical data.

In [ ]:
# Cluster at 0.99 threshold

clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    predictions, threshold_match_probability=0.99
)
df_clusters = clusters.as_pandas_dataframe()

print("Total rows:", df_clusters.shape[0])
print("Unique clusters:", df_clusters['cluster_id'].nunique())
print()

cluster_sizes = df_clusters['cluster_id'].value_counts()
print("Cluster size distribution:")
print(cluster_sizes.value_counts().sort_index())

INFO:splink.internals.connected_components:Completed iteration 1, num edges remaining to process: 0


Total rows: 47072
Unique clusters: 46990

Cluster size distribution:
count
1    46921
2       60
3        7
4        1
6        1
Name: count, dtype: int64


**Note:**
With 0.95, Unique cluster: 46,585
while for 0.99, Unique cluster: 46,990

Less rows clustered.

**10: Quality checks**

In [ ]:
# 10a: Inspect the largest cluster

biggest_cluster_id = cluster_sizes.idxmax()
print("Largest cluster ID:", biggest_cluster_id,
      "— size:", cluster_sizes.max())
print()
df_clusters[df_clusters['cluster_id'] == biggest_cluster_id][
    ['persona_idno', 'persona_type', 'name', 'lastname',
     'gender_clean', 'birth_date', 'birth_place']
]


Largest cluster ID: persona-24351 — size: 6



,persona_idno,persona_type,name,lastname,gender_clean,birth_date,birth_place
24213,persona-24351,godmother,anjelina,manrique,None,NaT,None
24221,persona-24359,godmother,anjelina,manrique,None,NaT,None
24225,persona-24363,godmother,anjelina,manrique,None,NaT,None
24581,persona-24719,godfather,anjelina,manrique,None,NaT,None
24600,persona-24738,mother,anjelina,manrique,None,NaT,None
42673,persona-42913,mother,anjelina,manrique,None,NaT,None


In [ ]:
# 10b: Check what proportion of multi-row clusters have date/place support ===

multi_clusters = cluster_sizes[cluster_sizes > 1].index
multi_df = df_clusters[df_clusters['cluster_id'].isin(multi_clusters)]

support = multi_df.groupby('cluster_id').agg(
    has_date  = ('birth_date',  lambda x: x.notna().any()),
    has_place = ('birth_place', lambda x: x.notna().any()),
)

name_only    = ((~support['has_date']) & (~support['has_place'])).sum()
with_support = len(support) - name_only

print("Total multi-row clusters :", len(support))
print("Name-only (no date/place):", name_only,
      f"({round(name_only / len(support) * 100, 1)}%)")
print("Has date or place support:", with_support)

Total multi-row clusters : 69
Name-only (no date/place): 41 (59.4%)
Has date or place support: 28


**11: Export**

In [ ]:

df_clusters.to_csv("personas_clustered_iter3.csv", index=False)

from google.colab import files
files.download("personas_clustered_iter3.csv")

print("Export complete.")
print("Rows exported   :", df_clusters.shape[0])
print("Unique clusters :", df_clusters['cluster_id'].nunique())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Export complete.
Rows exported   : 47072
Unique clusters : 46990


**Below steps is an extra, only for manual review exports**

**12. Export for manual review**

In [ ]:
# Only the merged rows

df = pd.read_csv("personas_clustered_iter3.csv", low_memory=False)
sizes = df['cluster_id'].value_counts()
dupes = df[df['cluster_id'].isin(sizes[sizes > 1].index)].copy()

In [ ]:
# Rows of the same cluster must sit together for review

dupes = dupes.sort_values(['cluster_id', 'event_idno'])

In [ ]:
# Keep selected columns for report review

cols = ['cluster_id','persona_idno','event_idno','original_identifier',
        'persona_type','name','lastname','gender_clean',
        'birth_date','birth_place','resident_in','death_date']
dupes = dupes[cols]

In [ ]:
# Evidence flag- Mark whether each cluster has anything beyond name agreement

support = dupes.groupby('cluster_id').agg(
    has_support=('birth_date', lambda x: x.notna().any())
)
# or combine birth_date/birth_place: (x.notna().any()) for either column
dupes['evidence'] = dupes['cluster_id'].map(
    support['has_support'].map({True:'name+date/place', False:'name-only'}))

In [ ]:
# Cluster size, handy for the reviewer
dupes['cluster_size'] = dupes['cluster_id'].map(sizes)

# Empty columns for the reviewer's verdict
dupes['reviewer_verdict'] = ''   # Correct / Incorrect / Uncertain
dupes['reviewer_notes'] = ''

dupes.to_csv("duplicates_for_review.csv", index=False)

print("Clusters for review :", dupes['cluster_id'].nunique())
print("Records involved    :", len(dupes))
print("Name-only clusters  :", (support == False).sum())
print("With date/place     :", (support == True).sum())

Clusters for review : 69
Records involved    : 151
Name-only clusters  : has_support    43
dtype: int64
With date/place     : has_support    26
dtype: int64


**12b: Pairwise evidence**

In [ ]:
# 12b: Pairwise evidence — the >=0.99 pairs behind the clusters
# Answers "on what basis were these two records merged": both records'
# fields side by side, plus the model's match probability.

pairs_evidence = df_predictions[df_predictions['match_probability'] >= 0.99].copy()

evidence_cols = ['match_probability', 'match_weight',
                 'persona_idno_l', 'persona_idno_r',
                 'name_l', 'name_r',
                 'lastname_l', 'lastname_r',
                 'gender_clean_l', 'gender_clean_r',
                 'birth_date_l', 'birth_date_r',
                 'birth_place_l', 'birth_place_r']
evidence_cols = [c for c in evidence_cols if c in pairs_evidence.columns]

pairs_evidence = (pairs_evidence[evidence_cols]
                  .sort_values('match_probability', ascending=False))
pairs_evidence['match_probability'] = pairs_evidence['match_probability'].round(4)

pairs_evidence.to_csv("merged_pairs_evidence.csv", index=False)
print("Pairs at >=0.99:", len(pairs_evidence))   # expect ~102

Pairs at >=0.99: 102


In [ ]:
# 12c: Download both files (Colab)
from google.colab import files
files.download("duplicates_for_review.csv")
files.download("merged_pairs_evidence.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>